# Sale Forecasting Task 2 - Kaggle Full Pipeline

Notebook này được tự động generate để chạy thẳng pipeline bạn đã xây dựng trên môi trường Kaggle.

### Hướng dẫn sử dụng:
1. **Tạo Dataset 1 (Code Repo)**: Upload file `kaggle_code.zip` (được tạo bởi `prepare_kaggle.sh`) lên Kaggle Dataset để import code vào notebook.
2. **Tạo Dataset 2 (Data)**: Upload 3 file parquet của BTC lên làm một Kaggle Dataset.
3. **Import vào Notebook**: Add 2 dataset trên vào notebook này (Data -> Add Data).
4. Sửa `YOUR_CODE_DATASET` và `YOUR_PARQUET_DATASET` ở Cell bên dưới tương ứng với tên thư mục trong `/kaggle/input/`.

In [ ]:
import os
import shutil
import yaml

# ==================================================================
# 1. ĐỔI TÊN THƯ MỤC NÀY THÀNH TÊN DATASET SAU KHI ADD VÀO KAGGLE:
# ==================================================================
CODE_DATASET_NAME = "sale-forecasting-code"       # Thư mục chứa config.yaml và src/
DATA_DATASET_NAME = "sale-forecasting-parquets"   # Thư mục chứa 3 file parquet

CODE_PATH = f"/kaggle/input/{CODE_DATASET_NAME}"
DATA_PATH = f"/kaggle/input/{DATA_DATASET_NAME}"

# Copy source code sang /kaggle/working/ vì /kaggle/input/ là read-only
if os.path.exists(CODE_PATH):
    os.system(f"cp -r {CODE_PATH}/src .")
    os.system(f"cp {CODE_PATH}/config.yaml .")
    os.system(f"cp {CODE_PATH}/requirements.txt .")
else:
    print(f"[CẢNH BÁO] Không tìm thấy {CODE_PATH}. Đảm bảo đã upload code dataset.")

# Điều chỉnh lại config.yaml để output ra /kaggle/working/
if os.path.exists('config.yaml'):
    with open('config.yaml', 'r') as f:
        cfg = yaml.safe_load(f)
    
    cfg['DATA_DIR']   = DATA_PATH
    cfg['OUTPUT_DIR'] = '/kaggle/working/outputs'
    cfg['MODEL_DIR']  = '/kaggle/working/models'
    cfg['REPORT_DIR'] = '/kaggle/working/reports'
    cfg['DEBUG_SAMPLE'] = False  # Chạy dữ liệu full trên máy mạnh của Kaggle
    
    with open('config.yaml', 'w') as f:
        yaml.dump(cfg, f)
    print("✅ Cập nhật config.yaml thành công cho môi trường Kaggle.")

In [ ]:
!pip install -q pyyaml pyarrow lightgbm scikit-learn

# Bước 1: Kiểm tra dữ liệu
!python src/data_audit.py

In [ ]:
# Bước 2: Baseline & Validation baseline (Khoảng 2-3 phút)
!python src/baseline.py
!python src/validation.py

In [ ]:
# Bước 3: Generate Features (Nặng nhất, mất 15-30 phút tuỳ CPU)
!python src/features.py

In [ ]:
# Bước 4: Train mô hình
!python src/train_lgbm.py

In [ ]:
# Bước 5: Blend các mô hình & Post process tối ưu
!python src/blend.py
!python src/postprocess.py

In [ ]:
# Bước 6: Kiểm tra submission cuối cùng
!python src/check_submission.py

import pandas as pd
sub = pd.read_csv('/kaggle/working/outputs/submission_final.csv')
display(sub.head())